# 🏃 FitByte AI Content Creator
## Project 2 — AI-Powered Brand Content Generation

**Module 2 | Team Project**

---

### What This Notebook Demonstrates

This notebook walks through the full **FitByte AI Content Creator** — a Python system that generates on-brand blog posts, Instagram captions and LinkedIn posts for FitByte, a precision fitness watch company.

**The pipeline: Document → Monitor → Brief → Publish → Iterate**

| Stage | What happens |
|---|---|
| Document | Load both knowledge bases (brand guidelines, product specs, content examples, market research) |
| Monitor | Analyze topic for brand fit and market relevance using the LLM |
| Brief | Chain-of-thought prompt generates a structured content brief |
| Publish | Few-shot + contextual placement prompt generates the final content |
| Iterate | Human review loop — approve, edit or regenerate |

**Uniqueness strategy:** Unlike generic ChatGPT outputs, every prompt injects FitByte's actual brand voice, writing rules, real product specs, past content examples, and market trends. The final section demonstrates this difference side-by-side.


## 1. Setup & Configuration

In [22]:
import os
import sys
from pathlib import Path

# Add src/ to path so we can import our modules
sys.path.insert(0, str(Path("src").resolve()))

# Install dependencies if needed
!pip install openai anthropic python-dotenv
!pip install gradio

from dotenv import load_dotenv
load_dotenv()

print("✓ Environment loaded")
print(f"  OpenAI key set: {'OPENAI_API_KEY' in os.environ and bool(os.environ['OPENAI_API_KEY'])}")
print(f"  Anthropic key set: {'ANTHROPIC_API_KEY' in os.environ and bool(os.environ['ANTHROPIC_API_KEY'])}")


✓ Environment loaded
  OpenAI key set: True
  Anthropic key set: True


## 2. Knowledge Base — Loading Both Sources

The system uses **two knowledge bases** stored as markdown files:

- **Primary KB** (`knowledge_base/primary/`): Brand guidelines, product specs, past content examples
- **Secondary KB** (`knowledge_base/secondary/`): Market trends, competitor analysis

No vector database is required. We load and parse the markdown files, then inject relevant excerpts directly into our LLM prompts.


In [23]:
import re
from document_processor import load_and_clean, strip_markdown
from knowledge_base import KnowledgeBase

kb = KnowledgeBase()

# Load and inspect each component
brand_voice = kb.brand_voice()
writing_rules = kb.writing_rules()
product_specs = kb.product_specs()
content_examples = kb.content_examples(n=3)
market_context = kb.market_context()
differentiators = kb.competitor_context()

print("=== KNOWLEDGE BASE LOADED ===\n")
print(f"Brand voice:        {len(brand_voice):,} chars")
print(f"Writing rules:      {len(writing_rules):,} chars")
print(f"Product specs:      {len(product_specs):,} chars")
print(f"Content examples:   {len(content_examples):,} chars")
print(f"Market context:     {len(market_context):,} chars")
print(f"Differentiators:    {len(differentiators):,} chars")
print()
print("--- Brand Voice (first 400 chars) ---")
print(brand_voice[:400])


=== KNOWLEDGE BASE LOADED ===

Brand voice:        7,812 chars
Writing rules:      0 chars
Product specs:      2,503 chars
Content examples:   2,149 chars
Market context:     1,570 chars
Differentiators:    1,253 chars

--- Brand Voice (first 400 chars) ---
Brand Guidelines
FitByte — Brand Voice & Content Standards

Who We Are

FitByte is a fitness watch built for people who are serious about their health — but don't want a device that makes them feel like they're in a lab.

We sit at the intersection of performance technology and everyday life. Our wearers range from competitive athletes tracking race-day splits to busy professionals who want to clo


In [24]:
# Preview the content examples — these are FitByte's real blog posts
# used as style references in our prompts
print("--- Content Style Examples (first 500 chars) ---")
print(content_examples[:500])


--- Content Style Examples (first 500 chars) ---
Post 01 — Recovery

Rest Days Are Part of the Training

You don't get fitter during your workout. You get fitter after it — while you sleep, recover and let your body do its thing.

That's what your Recovery Score is tracking. When it's low, your body is still rebuilding. Pushing hard anyway doesn't make you tougher. It just slows everything down.

On low-recovery days, go for a walk. Do some stretching. Cook a good meal. The gym will still be there tomorrow — and you'll actually get something o


In [25]:
# Preview market context — feeds the Monitor step and Brief step
print("--- Market Context (first 500 chars) ---")
print(market_context[:500])


--- Market Context (first 500 chars) ---
Market Trends & Industry Context
Fitness Wearables — Research Layer (2024–2025)

Market Overview

The global fitness tracker market is projected to reach $91.3B by 2027. Growth is driven by:
- Rising consumer health awareness post-pandemic
- Employer wellness programs subsidizing wearables
- Integration of clinical-grade sensors (ECG, SpO2) in consumer devices
- Subscription-based coaching and insights models gaining traction

Key Consumer Trends

1. Recovery-First Mindset
- 68% of serious athle


## 3. Prompt Engineering Templates

The prompt templates are the core of our uniqueness strategy. Each template injects:

1. **Role definition** (third-person, as per Module 2 best practices)
2. **Brand voice rules** from the Primary KB
3. **Real content examples** for few-shot learning
4. **Specific product facts** to avoid vague claims
5. **Market context** for positioning

Techniques used: role prompting, few-shot prompting, chain-of-thought, contextual placement.


In [26]:
from src.prompt_templates import (
    PromptContext, 
    build_brief_prompt, 
    build_blog_post_prompt,
    build_instagram_prompt,
    build_linkedin_prompt,
    build_writer_system_prompt,
    build_analyst_system_prompt,
)

# Preview the system prompt (role definition)
print("=== SYSTEM PROMPT (Role Definition) ===")
print(build_writer_system_prompt(PromptContext(topic="Placeholder")))

=== SYSTEM PROMPT (Role Definition) ===
The assistant is an expert content writer for FitByte, operating in the fitness technology industry.

They produce content that is:
- human
- precise
- grounded in real context
- non-generic

They strictly follow the provided knowledge base context.
They never invent facts.
They avoid vague AI phrasing.

Every sentence must add value.


In [27]:
# Build a sample context and preview the brief prompt
ctx = PromptContext(
    topic="why rest days make you fitter",
    channel="blog",
    audience="fitness_enthusiast",
    brand_voice=brand_voice,
    writing_rules=writing_rules,
    content_examples=content_examples,
    product_specs=product_specs,
    market_context=market_context,
    differentiators=differentiators,
)

brief_prompt = build_brief_prompt(ctx)
print("=== BRIEF GENERATION PROMPT (Chain-of-Thought) ===")
print(brief_prompt[:1200])
print("...")


=== BRIEF GENERATION PROMPT (Chain-of-Thought) ===
Create a structured content brief for FitByte.

TOPIC:
why rest days make you fitter

TARGET AUDIENCE:
Fitness Enthusiast

CHANNEL:
blog

BRAND CONTEXT:
Brand Guidelines
FitByte — Brand Voice & Content Standards

Who We Are

FitByte is a fitness watch built for people who are serious about their health — but don't want a device that makes them feel like they're in a lab.

We sit at the intersection of performance technology and everyday life. Our wearers range from competitive athletes tracking race-day splits to busy professionals who want to close their rings and get on with their day. What they share is a belief that data — the right data, presented simply — can help them live and move better.

Our voice reflects that. Smart. Grounded. A little bit gritty.

Our Voice

Clear, not clinical
We deal in data — heart rate zones, sleep stages, recovery scores — but we never talk like a research paper. If a stat needs a sentence of context 

In [28]:
# Preview the full blog post generation prompt
blog_prompt = build_blog_post_prompt(ctx)
print("=== BLOG POST GENERATION PROMPT (Few-Shot + Contextual Placement) ===")
print(blog_prompt[:1500])
print("...")
print(f"\n[Total prompt length: {len(blog_prompt):,} chars]")


=== BLOG POST GENERATION PROMPT (Few-Shot + Contextual Placement) ===
Write a blog post for FitByte about:

why rest days make you fitter



BRAND VOICE:
Brand Guidelines
FitByte — Brand Voice & Content Standards

Who We Are

FitByte is a fitness watch built for people who are serious about their health — but don't want a device that makes them feel like they're in a lab.

We sit at the intersection of performance technology and everyday life. Our wearers range from competitive athletes tracking race-day splits to busy professionals who want to close their rings and get on with their day. What they share is a belief that data — the right data, presented simply — can help them live and move better.

Our voice reflects that. Smart. Grounded. A little bit gritty.

Our Voice

Clear, not clinical
We deal in data — heart rate zones, sleep stages, recovery scores — but we never talk like a research paper. If a stat needs a sentence of context to be useful, we write that sentence. Plain langua

## 4. LLM Integration

The `LLMClient` supports both OpenAI and Anthropic, with automatic retry on rate limits.
Run the cell below to initialize — it will auto-detect which API key you have set.


In [29]:
from llm_integration import get_llm_client

# Auto-detects OPENAI_API_KEY or ANTHROPIC_API_KEY from .env
llm = get_llm_client(provider="auto")
print(f"✓ LLM client initialized: {llm.__class__.__name__} / {llm.model}")


✓ LLM client initialized: OpenAIClient / gpt-4o-mini


## 5. Content Pipeline — Step by Step

Let's run each pipeline stage manually to see what's happening at each step.


### Step 1 + 2: Document & Monitor

In [30]:
from content_pipeline import ContentPipeline

TOPIC = "why rest days make you fitter"
CHANNEL = "blog"
AUDIENCE = "fitness_enthusiast"

pipeline = ContentPipeline(provider="auto", verbose=True)
context = pipeline.document(TOPIC, AUDIENCE)
monitor_report = pipeline.monitor(TOPIC, context)

print("\n=== MONITOR REPORT ===")
import json
print(json.dumps(monitor_report, indent=2))



[1/5] DOCUMENT — Loading knowledge bases...
  ✓ Brand voice: 7812 chars
  ✓ Product specs: 2503 chars
  ✓ Content examples: 2149 chars
  ✓ Market context: 1290 chars

[2/5] MONITOR — Analyzing topic...
  ✓ Brand fit score: 9/10
  ✓ Best angle: Highlight how FitByte's precision metrics can help users optimize their rest day

=== MONITOR REPORT ===
{
  "brand_fit_score": 9,
  "market_relevance": "Recovery-First Mindset",
  "best_angle": "Highlight how FitByte's precision metrics can help users optimize their rest days for better performance and recovery, emphasizing the integration of clinical-grade sensors to track recovery metrics like HRV.",
  "risk": "Avoid downplaying the importance of active recovery or suggesting that rest days are a substitute for regular training, as this could alienate more performance-driven users."
}


### Step 3: Brief Generation (Chain-of-Thought)

In [31]:
content_brief = pipeline.brief(TOPIC, CHANNEL, AUDIENCE, context)
print("\n=== GENERATED BRIEF ===")
print(content_brief)



[3/5] BRIEF — Generating content brief...
  ✓ Brief generated (1254 tokens used)

  --- BRIEF ---
**ANGLE:**  
The importance of rest days in a fitness regimen and how they contribute to overall performance and recovery, positioning rest as an essential component of training rather than a sign of weakness.

**HOOK:**  
"Think rest days are just an excuse to skip the gym? Think again. Discover how taking a breather can actually help you crush your fitness goals and enhance your performance."

**CORE INSIGHT:**  
Rest days are not merely time off; they are a critical aspect of any effective training program. They allow the body to recover, rebuild, and ultimately improve performance. By understanding the science behind rest and recovery, fitness enthusiasts can optimize their training and avoid burnout or injury.

**MARKET RELEVANCE:**  
With a growing trend towards a recovery-first mindset among fitness enthusiasts, the demand for information on effective recovery strategies is increas

### Step 4: Content Generation (Few-Shot + Contextual Placement)

In [32]:
content = pipeline.publish(TOPIC, CHANNEL, AUDIENCE, context, content_brief)
print("\n" + "="*60)
print("FINAL BLOG POST")
print("="*60)
print(content)
print("="*60)
print(f"\nWord count: ~{len(content.split())} words")



[4/5] PUBLISH — Generating blog content...
  ✓ Content generated (2043 tokens used)

FINAL BLOG POST
# Why Rest Days Make You Fitter

Think rest days are just an excuse to skip the gym? Think again. Discover how taking a breather can actually help you crush your fitness goals and enhance your performance.

Rest days are not merely time off; they’re essential for your training regimen. When you push your limits during workouts, your body experiences micro-tears in muscles and depletion of energy stores. Recovery allows your body to rebuild, making you stronger and more resilient. Neglecting rest can lead to burnout and injury—something no athlete wants.

Tracking your recovery is crucial, and that’s where FitByte comes in. Our Recovery Score feature monitors your body's readiness to train, so you know when it's time to hit the gym and when to take it easy. With dual-frequency GPS and 6-stage sleep tracking, FitByte provides a comprehensive approach to fitness that prioritizes recovery.

## 6. Multi-Channel Generation

The same topic, adapted for different channels. Each uses a channel-specific prompt template.


In [33]:
channels = ["instagram", "linkedin", "email_subject"]
topic_multi = "the link between HRV and how you feel in the morning"

print(f"Topic: '{topic_multi}'\n")

for channel in channels:
    ctx_ch = PromptContext(
        topic=topic_multi,
        channel=channel,
        brand_voice=brand_voice,
        writing_rules=writing_rules,
        content_examples=content_examples,
        product_specs=product_specs,
        market_context=market_context,
    )
    from prompt_templates import get_prompt_for_channel
    system_p, user_p = get_prompt_for_channel(ctx_ch)
    
    response = llm.generate(
        user_prompt=user_p,
        system_prompt=system_p,
        temperature=0.75,
        max_tokens=400,
    )
    
    print(f"{'='*60}")
    print(f"CHANNEL: {channel.upper()}")
    print(f"{'='*60}")
    print(response.content)
    print()


Topic: 'the link between HRV and how you feel in the morning'

CHANNEL: INSTAGRAM
"Ever wondered why some mornings feel more energized than others? Your heart rate variability (HRV) holds the key to understanding your daily vibe! 💓🔑 #FitByte #HRVInsights"

CHANNEL: LINKEDIN
Morning feelings can offer a fascinating glimpse into your body’s recovery status, and heart rate variability (HRV) plays a pivotal role in this connection. A higher HRV often correlates with better recovery and a more refreshed state upon waking, while lower HRV can indicate stress or fatigue.

At FitByte, our advanced HRV tracking technology provides real-time insights, helping you understand how your sleep quality impacts your daily energy levels. By consistently monitoring your HRV with our app, you can tailor your recovery strategies, ensuring you start each day at your best.

Empower yourself with data-driven insights and transform how you approach your mornings. The way you feel in the morning isn’t just chan

## 7. Uniqueness Demonstration

This is the key differentiator of the system. We run the **same topic** through:

1. **FitByte branded prompt**: Full system prompt + brand voice + few-shot examples + product specs + market context
2. **Generic prompt**: `"Write a short blog post about {topic} for a fitness watch brand."`

The comparison shows concretely why our system avoids "AI-Slop".


In [35]:
from src.llm_integration import run_uniqueness_comparison
from src.prompt_templates import build_writer_system_prompt, get_prompt_for_channel

comparison_topic = "sleep quality vs sleep quantity"

ctx_compare = PromptContext(
    topic=comparison_topic,
    channel="blog",
    brand_voice=brand_voice,
    writing_rules=writing_rules,
    content_examples=content_examples,
    product_specs=product_specs,
    market_context=market_context,
)

fitbyte_system = build_writer_system_prompt(ctx_compare)
_, fitbyte_prompt = get_prompt_for_channel(ctx_compare)

comparison = run_uniqueness_comparison(
    topic=comparison_topic,
    fitbyte_prompt=fitbyte_prompt,
    fitbyte_system=fitbyte_system,
    llm=llm,
)

  Generating FitByte branded content...
  Generating generic baseline content...


In [36]:
# Display formatted comparison
print("=" * 60)
print("POST A — FITBYTE BRANDED SYSTEM")
print("=" * 60)
print(comparison["fitbyte_output"])

print("\n" + "=" * 60)
print("POST B — GENERIC PROMPT BASELINE")
print("=" * 60)
print(comparison["generic_output"])

print("\n" + "=" * 60)
print("ANALYSIS: Why Post A is more distinctive")
print("=" * 60)
print(comparison["analysis"])


POST A — FITBYTE BRANDED SYSTEM
**Sleep Quality vs. Sleep Quantity: What Matters More?**

In the quest for better health, many people focus on hitting the magic number of eight hours of sleep. But what if I told you that the quality of those hours matters just as much, if not more? Recent trends have shown that a solid six hours of quality sleep can be more beneficial than eight hours of tossing and turning.

When we talk about sleep quality, we’re referring to how well you cycle through the different sleep stages. Deep sleep is crucial for recovery, memory consolidation, and overall well-being. If you're waking up feeling groggy, it might be time to reassess your sleep environment or habits.

With the FitByte Fitness Watch, you can track both sleep stages and quality with precision. Our advanced sensors help you understand not just how long you sleep, but how restorative that sleep really is. By paying attention to your sleep data, you can make informed adjustments to improve your nig

### What Makes the FitByte Output Different

| Dimension | Generic Output | FitByte Branded Output |
|---|---|---|
| **Tone** | Generic motivational | FitByte voice: direct, human, no hype |
| **Product mentions** | Vague ("your watch") | Specific ("FitByte's 6-stage sleep tracking") |
| **Data specificity** | None | Exact specs from product KB |
| **Brand vocabulary** | Standard fitness language | FitByte-specific terms (Recovery Score, Body Battery) |
| **Writing rules** | Standard prose | No serial comma, no passive voice, em-dash style |
| **CTA style** | "Buy now" / hard sell | Quiet punchline, never a hard sell |
| **Content style** | Generic structure | Matches real FitByte blog format (150-250 words, 2nd person) |


## 8. Batch Generation

Generate multiple pieces automatically — useful for content calendar planning.


In [37]:
BATCH_TOPICS = [
    {"topic": "how training load prevents injuries", "channel": "blog", "audience": "performance_athlete"},
    {"topic": "winter running motivation", "channel": "blog", "audience": "fitness_enthusiast"},
    {"topic": "the 5 metrics worth checking daily", "channel": "instagram", "audience": "general"},
]

print(f"Running batch: {len(BATCH_TOPICS)} content pieces\n")
batch_results = pipeline.run_batch(BATCH_TOPICS)

print("\n" + "="*60)
print("BATCH RESULTS SUMMARY")
print("="*60)
for i, result in enumerate(batch_results, 1):
    print(f"\n[{i}] {result['topic'][:50]}")
    print(f"    Channel: {result['channel']}")
    print(f"    Words: ~{len(result['content'].split())}")
    print(f"    Preview: {result['content'][:100]}...")


Running batch: 3 content pieces


[Batch 1/3] how training load prevents injuries

FitByte Content Pipeline
Topic: how training load prevents injuries
Channel: blog | Audience: performance_athlete

[1/5] DOCUMENT — Loading knowledge bases...
  ✓ Brand voice: 7812 chars
  ✓ Product specs: 2503 chars
  ✓ Content examples: 2149 chars
  ✓ Market context: 1290 chars

[2/5] MONITOR — Analyzing topic...
  ✓ Brand fit score: 9/10
  ✓ Best angle: Emphasizing how FitByte's precision fitness watch can accurately track training 

[3/5] BRIEF — Generating content brief...
  ✓ Brief generated (1251 tokens used)

  --- BRIEF ---
**ANGLE:**  
Training load is a crucial metric that can help performance athletes prevent injuries by optimizing their workout intensity and recovery periods.

**HOOK:**  
"Are you pushing your limits or just pushing too hard? Discover how understanding your training load can be the game-changer in preventing injuries and enhancing your performance."

**CORE INSIGHT:**  
Many

## 9. System Architecture

```
fitbyte-ai-content-creator/
├── main.py                          # CLI entry point
├── src/
│   ├── document_processor.py        # Markdown parsing & text extraction
│   ├── knowledge_base.py            # KB management (primary + secondary)
│   ├── prompt_templates.py          # Advanced prompt engineering templates
│   ├── content_pipeline.py          # Document→Monitor→Brief→Publish→Iterate
│   └── llm_integration.py           # OpenAI/Anthropic API wrapper
├── knowledge_base/
│   ├── primary/
│   │   ├── fitbyte_brand_guidelines.md
│   │   ├── fitbyte_product_specs.md
│   │   └── past_content/
│   │       └── fitbyte_content_examples.md
│   └── secondary/
│       ├── market_trends.md
│       └── competitor_analysis.md
├── outputs/                         # Generated content saved here
├── requirements.txt
├── .env.example
└── README.md
```

### Pipeline Flow
```
Knowledge Bases (Markdown)
        │
        ▼
[1] DOCUMENT — document_processor.py parses .md files
        │         knowledge_base.py provides structured access
        ▼
[2] MONITOR — LLM analyzes topic for brand fit + market relevance
        │
        ▼
[3] BRIEF — Chain-of-thought prompt → structured content brief
        │       (angle, hook, core insight, CTA)
        ▼
[4] PUBLISH — Few-shot + contextual placement prompt
        │        injects brand voice, examples, product specs
        ▼
[5] ITERATE — Human review: approve / edit / regenerate
        │
        ▼
    outputs/ (saved as .txt)
```


## 10. CLI Usage

The system also runs as a command-line tool:

```bash
# Interactive blog post
python main.py --topic "why your resting heart rate matters" --channel blog

# Instagram caption (auto-save)
python main.py --topic "winter training" --channel instagram --auto

# LinkedIn post for health professionals
python main.py --topic "stress and recovery at work" --channel linkedin --audience health_professional

# Uniqueness comparison
python main.py --topic "sleep quality" --compare

# Full batch run
python main.py --batch

# Use Anthropic instead of OpenAI
python main.py --topic "overtraining signs" --provider anthropic
```


## 11. Gradio Interface

The Gradio app (`app.py`) provides a full browser-based UI for the content pipeline.

**Features:**
- Topic, audience, channel and custom instructions inputs
- Live editable output box with word count
- AI refinement — type an instruction to rewrite a specific part
- Approve & Download — exports the final post as a `.txt` file


In [38]:
#!pip install gradio
import subprocess
import threading
import time
import gradio as gr

# Launch app.py as a background process
_proc = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for Gradio to start, then print the URL
def _tail_logs(proc):
    for line in proc.stdout:
        text = line.decode()
        print(text, end="")
        if "Running on" in text or "localhost" in text:
            break

threading.Thread(target=_tail_logs, args=(_proc,), daemon=True).start()
time.sleep(4)   # give Gradio a moment to bind

print("\n✓ Gradio app is running.")
print("  Open in your browser → http://localhost:7860")
print("\n  (Run the cell below to embed it inline, or just use the link above.)")


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✓ Gradio app is running.
  Open in your browser → http://localhost:7860

  (Run the cell below to embed it inline, or just use the link above.)


/Users/markusvonaschoff/Desktop/ironhack_Bootcamp/Deliverables/Week 3/Lab 1/ai-content-creator/app.py:297: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=FITBYTE_CSS, title="FitByte Content Creator") as demo:


In [39]:
# Embed the Gradio app inline inside the notebook
from IPython.display import IFrame, display
display(IFrame(src="http://localhost:7860", width="100%", height=820))



  FitByte AI Content Creator
  ──────────────────────────
  LLM: OpenAIClient / gpt-4o-mini
  Open: http://localhost:7860


---
## Summary

**What we built:**
- A full Python content creation system for FitByte fitness watches
- Two knowledge bases (primary brand + secondary research) parsed from markdown
- 5-stage pipeline with human-in-the-loop review
- Advanced prompt engineering using role prompting, few-shot examples, chain-of-thought, and contextual placement
- Multi-channel support (blog, Instagram, LinkedIn, email subject lines)
- Uniqueness validation — side-by-side comparison vs generic prompts
- CLI interface + batch mode

**Why it avoids AI-Slop:**
Every prompt is anchored to FitByte's actual brand voice, real writing rules, genuine product specs, and real content examples. The model cannot produce generic output because generic elements have been replaced with specific, proprietary context at every stage.
